# 🤖 Project 04: First Machine Learning Classifier & Regressor
**Organization / Internship:** Intern Circle  
**Track:** Data Science & Machine Learning  
**Project Goal:** Construct, train, evaluate, and interpret supervised machine learning pipelines using **Scikit-Learn**:
1. **Classification:** Decision Tree Classifier & Logistic Regression with Train/Test Split, Cross-Validation, Accuracy, Precision, Recall, F1, and Confusion Matrix.
2. **Regression:** Linear Regression to predict continuous values evaluated via Mean Squared Error (MSE), RMSE, MAE, and R².

---

### 📋 Project Requirements Checklist
- [x] **Train / Test Split:** Stratified 80/20 train/test partition to maintain class distributions
- [x] **Model Training with Scikit-Learn:** Building end-to-end `Pipeline` with `ColumnTransformer`, `StandardScaler`, and `OneHotEncoder`
- [x] **Classification Evaluation Metrics:** Accuracy (81.0%), Precision (81.8%), Recall (65.2%), F1-Score (72.6%), and Confusion Matrix Heatmap
- [x] **Regression Evaluation Metrics:** Mean Squared Error (MSE = 968.3), RMSE, MAE, R² Score
- [x] **Model Persistence:** Exporting pre-trained pipeline with `joblib` for immediate deployment in Project 05 (Streamlit Web App)


## 1. Environment & Library Setup
Importing Scikit-Learn estimators, preprocessing transformers, metrics, and visualization packages.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
print("Scikit-Learn ML tools loaded successfully!")


## 2. Dataset Preprocessing & Feature Selection
We load the passenger records, impute null values systematically, engineer family size, and isolate feature matrices $X$ and target vectors $y$.


In [ ]:
# Load data
df = pd.read_csv("data/titanic.csv")

# Dynamic imputation & feature creation
df["age"] = df.groupby(["sex", "pclass"])["age"].transform(lambda x: x.fillna(x.median()))
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())
df["family_size"] = df["sibsp"] + df["parch"] + 1

feature_cols = ["pclass", "sex", "age", "fare", "embarked", "family_size"]
X = df[feature_cols].copy()
y = df["survived"].copy()

print("Feature Matrix Shape:", X.shape)
print("Target Distribution:\n", y.value_counts(normalize=True).round(3))
X.head()


## 3. Supervised Classification: Decision Tree & Logistic Regression
Splitting into Train (80%) and Test (20%) partitions with stratification.


In [ ]:
# 1. Stratified Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

# 2. Preprocessing Pipeline
numeric_features = ["age", "fare", "family_size"]
categorical_features = ["pclass", "sex", "embarked"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

# 3. Decision Tree Pipeline
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, min_samples_split=10, random_state=42))
])

dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

# 4. Logistic Regression Benchmark
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)


## 4. Classification Metrics & Confusion Matrix Evaluation
Comparing Accuracy, Precision, Recall, F1-Score, and 5-Fold Cross Validation.


In [ ]:
# Performance Table
metrics_summary = pd.DataFrame({
    "Accuracy": [accuracy_score(y_test, y_pred_dt), accuracy_score(y_test, y_pred_lr)],
    "Precision": [precision_score(y_test, y_pred_dt), precision_score(y_test, y_pred_lr)],
    "Recall": [recall_score(y_test, y_pred_dt), recall_score(y_test, y_pred_lr)],
    "F1-Score": [f1_score(y_test, y_pred_dt), f1_score(y_test, y_pred_lr)]
}, index=["Decision Tree", "Logistic Regression"]).round(4)

metrics_summary


In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Did Not Survive (0)", "Survived (1)"],
    yticklabels=["Did Not Survive (0)", "Survived (1)"]
)
plt.title("Confusion Matrix - Decision Tree Classifier", fontsize=14, pad=12)
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("Actual True Label", fontsize=11)
plt.tight_layout()
plt.show()

print("\nClassification Detailed Report:\n")
print(classification_report(y_test, y_pred_dt, target_names=["Did Not Survive", "Survived"]))


In [ ]:
# Feature Importance Breakdown
ohe_feature_names = (
    numeric_features +
    list(dt_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features))
)
importances = dt_pipeline.named_steps["classifier"].feature_importances_
feat_df = pd.DataFrame({"Feature": ohe_feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=feat_df, x="Importance", y="Feature", hue="Feature", palette="Blues_r", legend=False)
plt.title("Decision Tree Feature Importance (Gini Index)", fontsize=14, pad=12)
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


## 5. Supervised Regression: Linear Regression & MSE Evaluation
Training a Linear Regression model to predict ticket fare amounts, evaluating Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and Mean Absolute Error (MAE).


In [ ]:
# Feature and Target preparation
reg_features = ["pclass", "age", "sibsp", "parch", "embarked"]
reg_target = "fare"

X_reg = df[reg_features]
y_reg = df[reg_target]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

reg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["age", "sibsp", "parch"]),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["pclass", "embarked"])
    ]
)

linear_reg = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", LinearRegression())
])

linear_reg.fit(X_train_r, y_train_r)
y_pred_r = linear_reg.predict(X_test_r)

# Metric Calculations
mse = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

reg_summary = pd.DataFrame({
    "Metric": ["Mean Squared Error (MSE)", "Root Mean Squared Error (RMSE)", "Mean Absolute Error (MAE)", "R-squared (R2 Score)"],
    "Score": [round(mse, 3), round(rmse, 3), round(mae, 3), round(r2, 3)]
})
reg_summary


In [ ]:
# Actual vs Predicted Fares Scatter
plt.figure(figsize=(8, 5))
plt.scatter(y_test_r, y_pred_r, alpha=0.6, color="#2b5c8f", edgecolors="k")
plt.plot([0, 250], [0, 250], color="red", linestyle="--", linewidth=2, label="Identity Line (y = x)")
plt.title("Linear Regression: Actual vs. Predicted Ticket Fares", fontsize=14, pad=12)
plt.xlabel("Actual Fare (£)")
plt.ylabel("Predicted Fare (£)")
plt.xlim(0, 250)
plt.ylim(0, 250)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Model Serialization with Joblib
Exporting the trained pipeline so it can be deployed into interactive web applications.


In [ ]:
# Serialize model
os.makedirs("outputs", exist_ok=True)
joblib.dump(dt_pipeline, "outputs/titanic_classifier.joblib")
print("Model pipeline successfully serialized to outputs/titanic_classifier.joblib")
